**This notebook was copied and modified from the [original notebook](https://github.com/claireyung/mom6-panAn-iceshelf-tools/blob/7af8592655876530a919be9a6d89e3505a74aced/generate-draft/Generate-Charrassin-bathy.ipynb) in [claireyung/mom6-panAn-iceshelf-tools](https://github.com/claireyung/mom6-panAn-iceshelf-tools) (commit `7af8592`).**

**The changes replace the bilinear regridding of the Charrassin 2025 bed and ice thickness with binning (averaging) of the 500 m Charrassin pixels in each model cell. Bilinear regridding samples Charrassin only at each model cell centre, so a cell could get the depth of a narrow feature, e.g. a shallow fringe a few hundred metres wide at the coast or ice front. This created isolated shallow cells next to much deeper water, which caused the model instabilities described in [ACCESS-NRI/access-om3-configs#1855](https://github.com/ACCESS-NRI/access-om3-configs/pull/1855). The edits made are:**

- **New binning cell (`bin_charrassin`, after the model `topog` is loaded): assigns every Charrassin pixel to the model cell containing it and sums pixel counts (all, open ocean, wet = open ocean or floating ice, ice), bed elevation and ice thickness per cell. It reads the Charrassin GeoTIFFs directly, so the lat-lon netCDF conversions and the ESMF weights (hugemem job) are no longer needed.**
- **Bed: depth is the mean over wet pixels where at least half the cell is wet, otherwise the mean bed over all pixels: the bed under grounded ice, which the ice-shelf (cavity) topography uses to extend the ocean inland of the grounding line, and negative on land. The open-ocean and wet fractions are also saved.**
- **Ice thickness: the mean over pixels with ice where at least half the cell has ice, otherwise 0, so `thick > 0` still means at least half ice covered. The ice fraction is also saved.**
- **GEBCO fill: the GEBCO-based global topography is used only for cells not entirely within the Charrassin raster. Previously it was used wherever the regridded Charrassin elevation was >= 0, which also replaced Charrassin land (giving GEBCO depth minus land elevation) and Charrassin cells with an elevation of exactly 0 (giving an isolated deep cell surrounded by shallow cells).**
- **Output files keep their names and variables (`topog_Charrassin_regridded.nc`: `depth`, `lat`, `lon`; `ice_thickness_Charrassin_regridded.nc`: `thick`, `lat`, `lon`), plus the new fraction variables, and are written to `OUTPUT_DIR` (as float32, like the original files) with a `history` attribute recording the commit and a link to the exact version of this notebook used.**
- **New provenance cell (after the imports): records the git commit and builds the link, and stops if the notebook's code differs from the committed version. New `OUTPUT_DIR` cell.**
- **The surface and water height files are not regenerated (they are not used for the panan topography).**
- **All code cells not needed are commented out, so the notebook can be run top to bottom. Commented out `import cosima_cookbook` (not available in current analysis3 environments). Needs a compute node (several GB of memory, ~10-20 min).**

# Generate Charrassin bathy on 8km grid

copied from Ezhil: https://github.com/ezhilsabareesh8/Notebooks/blob/main/Regridd_Charrassin_topo_to_model_grid.ipynb

Data: https://datadryad.org/dataset/doi:10.5061/dryad.rbnzs7hkc


In [ ]:
import xesmf
import xarray as xr
import os
import cmocean as cm
import cartopy.crs as ccrs
# import cosima_cookbook as cc
import rasterio
from rasterio.windows import Window
from dask.distributed import Client
import dask.array as da
import numpy as np
from osgeo import gdal
import matplotlib.pyplot as plt

from pyproj import Transformer
from pyproj import CRS
ps_to_ll = Transformer.from_crs( "EPSG:3031","EPSG:4326", always_xy=True) 
#

In [ ]:
# Provenance: record the exact version of this notebook in the files it writes.
# The recorded commit only identifies the code that ran if the notebook's code is committed,
# so stop here if any cell's source differs from the committed version.
# (Unsaved edits in the open notebook can't be detected - save before running.)
import datetime
import getpass
import json
import subprocess
from pathlib import Path

import nci_ipynb

def _git(*args):
    res = subprocess.run(["git", *args], capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed: {res.stderr.strip()}")
    return res.stdout.strip()

nb_path = nci_ipynb.path().resolve()
repo_root = _git("-C", str(nb_path.parent), "rev-parse", "--show-toplevel")
NOTEBOOK = nb_path.relative_to(Path(repo_root).resolve()).as_posix()
commit = _git("-C", repo_root, "rev-parse", "HEAD")

def _cell_sources(nb_json):
    return [c["source"] for c in json.loads(nb_json)["cells"]]

with open(nb_path) as f:
    working = _cell_sources(f.read())
committed = _cell_sources(_git("-C", repo_root, "show", f"{commit}:{NOTEBOOK}"))
if working != committed:
    raise RuntimeError(f"{NOTEBOOK} differs from commit {commit}: commit (and push) your changes before running")

remote = _git("-C", repo_root, "remote", "get-url", "origin")
repo_url = remote.removesuffix(".git").replace("git@github.com:", "https://github.com/")
if not _git("-C", repo_root, "branch", "-r", "--contains", commit):
    print(f"WARNING: commit {commit} is not on any remote branch - push it so the link below works")

PROVENANCE = (f"{datetime.datetime.now().astimezone():%Y-%m-%d %H:%M:%S %Z}: created by {getpass.getuser()} "
              f"using {repo_url}/blob/{commit}/{NOTEBOOK}")
print(PROVENANCE)

In [ ]:
# Directory to write the output files to
OUTPUT_DIR = "/g/data/tm70/ds0092/model/inputs/access-om3/panan/2026.11.13"

```
Download: 
ICE_THICKNESS_ANTGG2022.tif
MASK_ANTGG2022.tif
BED_ANTGG2022.tif

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# # Open dataset
# dataset = gdal.Open('/g/data/x77/cy8964/Charrassin2025_Data/BED_ANTGG2022.tif')
# band = dataset.GetRasterBand(1)
# data = band.ReadAsArray()
#
# # Get GeoTransform and projection
# gt = dataset.GetGeoTransform()
# nx, ny = dataset.RasterXSize, dataset.RasterYSize
#
# # Build coordinates from geotransform
# x = np.arange(gt[0], gt[0] + nx * gt[1], gt[1])
# y = np.arange(gt[3], gt[3] + ny * gt[5], gt[5])
#
# # y is descending if gt[5] < 0 (common for north-up images), reverse if needed
# if gt[5] < 0:
#     y = y[::-1]
#     data = data[::-1, :]  # Flip data accordingly

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# # print("Driver: ", ds.GetDriver().ShortName)
# # print("Size: ", ds.RasterXSize, ds.RasterYSize)
# # print("Bands: ", ds.RasterCount)
# # print("Projection: ", ds.GetProjection())
# # print("GeoTransform: ", ds.GetGeoTransform())

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# da = xr.DataArray(
#     data,
#     dims=('y', 'x'),
#     coords={'x': x, 'y': y},
#     attrs={
#         'crs': dataset.GetProjection(),
#         'transform': gt,
#         'nodata': band.GetNoDataValue()
#     }
# )

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# ds_xr = xr.Dataset({'elevation': da})
#

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# BedMachine_v3 = xr.open_dataset('/g/data/x77/cy8964/satellite-meltrate/Adusumilli/BedMachineAntarctica-v3.nc')
# BedMachine_v3#.bed.plot()

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# da.plot(cmap='terrain')
# plt.title("Charrasin Bed elevation (meters) below sea level")
# #plt.savefig("/g/data1a/tm70/ek4684/Bedmachine_data/Charrasin_BED_elevation.png", dpi=300, bbox_inches='tight')
# plt.show()

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# # Create 2D coordinate grids
# X, Y = np.meshgrid(x, y)
#
# # Flatten and transform
# lon_flat, lat_flat = ps_to_ll.transform(X.ravel(), Y.ravel())
# lon_flat = (lon_flat + 280) % 360 - 280  # Normalize longitudes
#
# # Reshape to 2D
# new_longitudes = lon_flat.reshape(X.shape)
# new_latitudes = lat_flat.reshape(Y.shape)

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# ds_out = xr.Dataset(
#     data_vars={
#         "elevation": (("y", "x"), ds_xr.elevation.data)
#     },
#     coords={
#         "lat": (("y", "x"), new_latitudes),
#         "lon": (("y", "x"), new_longitudes)
#     },
#     attrs={
#         "title": "Bedmachine Antarctic Topography",
#         "source_crs": "EPSG:3031",
#         "converted_to": "EPSG:4326",
#         "note": "Original Charrassin Bed elevation data was in polar stereographic projection and converted to lat/lon."
#     }
# )
#
# ds_out["elevation"].attrs["units"] = "meters"
# ds_out["elevation"].attrs["long_name"] = "Surface elevation"
# ds_out["lat"].attrs["units"] = "degrees_north"
# ds_out["lon"].attrs["units"] = "degrees_east"
# #ds_out.to_netcdf("/g/data/x77/cy8964/Charrassin2025_Data/BED_ANTGG2022_lat_lon.nc")

### Now regrid

**[Out of date: these ESMF weights are no longer needed. The Charrassin data are now binned onto the model grid by `bin_charrassin` (below).]**

first create weights using interactive gadi job (to allow more memory)

this took ~10 mins

```

qsub -I -P x77 -q hugemem -l ncpus=192,mem=5880G,walltime=05:00:00,storage=gdata/hh5+gdata/ik11+gdata/x77+gdata/vk83
module load esmf/8.6.1 openmpi/4.1.7


mpirun -np 192 ESMF_RegridWeightGen -p none -i --ignore_degenerate -s /g/data/x77/cy8964/Charrassin2025_Data/BED_ANTGG2022_lat_lon.nc -d /g/data/x77/ahg157/inputs/mom6/global-8km/topog.nc -w /g/data/x77/cy8964/mom6/input/input-8km/Regridd_Charrassin_to_8km_global_grid_Bilinear.nc -m bilinear --netcdf4 --src_regional --check

In [ ]:
topog = xr.open_dataset('/g/data/x77/ahg157/inputs/mom6/global-8km/topog.nc', decode_times=False).rename({'geolon_t': 'lon', 'geolat_t': 'lat'})


In [ ]:
# Bin the 500 m Charrassin 2025 rasters (EPSG:3031, 13333 x 13333) onto the model grid.
# Every Charrassin pixel is assigned to the model cell that contains its centre, and per-cell statistics are summed.
# This replaces the bilinear regridding used previously, which sampled Charrassin only at each model cell centre, so a
# cell could get the depth of a narrow feature (e.g. a shallow fringe at the coast or ice front) unrepresentative of
# the cell. South of 40S the model grid is regular in longitude and latitude, so the cell edges are 1D.
CHARRASSIN_DIR = "/g/data/x77/cy8964/Charrassin2025_Data"
# panan model grid (identical to the southern 2885 supergrid rows of the global 8 km grid
# /g/data/x77/ahg157/inputs/mom6/global-8km/ocean_hgrid.nc, and reaching 37.5S)
hgrid = xr.open_dataset("/g/data/vk83/prerelease/configurations/inputs/access-om3/panan.4km/2026.01.08/ocean_hgrid.nc")
lat_edges_all = hgrid.y.isel(nxp=0).values[::2]
J_SOUTH = int(np.searchsorted(lat_edges_all, -40.0))  # number of model rows south of 40S
lon_edges = hgrid.x.isel(nyp=0).values[::2]  # cell edge longitudes, -280 to 80
lat_edges = lat_edges_all[: J_SOUTH + 1]
_xs = hgrid.x.isel(nyp=slice(0, 2 * J_SOUTH + 1)).values
_ys = hgrid.y.isel(nyp=slice(0, 2 * J_SOUTH + 1)).values
assert (_xs == _xs[0]).all() and (_ys == _ys[:, :1]).all(), "model grid is not rectilinear south of 40S"
del _xs, _ys
NX = lon_edges.size - 1
NCELL = J_SOUTH * NX


def bin_charrassin(block_rows=400):
    """Sum, for every model cell south of 40S, the number of Charrassin pixels in it (all, open ocean, wet = open ocean
    or floating ice, and ice = ice thickness > 0), the bed elevation over all and over wet pixels, and the ice
    thickness over ice pixels."""
    keys = ["n_all", "n_ocean", "n_wet", "n_ice", "bed_all", "bed_wet", "thick_ice"]
    sums = {k: np.zeros(NCELL) for k in keys}
    ps_to_ll = Transformer.from_crs("EPSG:3031", "EPSG:4326", always_xy=True)
    with rasterio.open(f"{CHARRASSIN_DIR}/BED_ANTGG2022.tif") as fbed, \
         rasterio.open(f"{CHARRASSIN_DIR}/MASK_ANTGG2022.tif") as fmask, \
         rasterio.open(f"{CHARRASSIN_DIR}/ICE_THICKNESS_ANTGG2022.tif") as fthick:
        T = fbed.transform
        assert fmask.transform == T and fthick.transform == T
        x_centres = T.c + T.a * (np.arange(fbed.width) + 0.5)
        for r0 in range(0, fbed.height, block_rows):
            nr = min(block_rows, fbed.height - r0)
            win = Window(0, r0, fbed.width, nr)
            bed = fbed.read(1, window=win).astype(np.float64).ravel()
            mask = fmask.read(1, window=win).ravel()
            thick = fthick.read(1, window=win).astype(np.float64).ravel()
            thick[~(thick < 1e30)] = 0.0  # nodata
            y_centres = T.f + T.e * (np.arange(r0, r0 + nr) + 0.5)
            X, Y = np.meshgrid(x_centres, y_centres)
            lon, lat = ps_to_ll.transform(X.ravel(), Y.ravel())
            lon = (lon + 280) % 360 - 280  # model longitude range
            i = np.searchsorted(lon_edges, lon, side="right") - 1
            j = np.searchsorted(lat_edges, lat, side="right") - 1
            ok = (i >= 0) & (i < NX) & (j >= 0) & (j < J_SOUTH)
            idx = (j * NX + i)[ok]
            bed, mask, thick = bed[ok], mask[ok], thick[ok]
            wet = (mask == 0) | (mask == 3)  # mask: ocean (0), ice-free land (1), grounded ice (2), floating ice (3), Lake Vostok (4)
            ice = thick > 0
            for k, w in [("n_all", None), ("n_ocean", (mask == 0).astype(float)), ("n_wet", wet.astype(float)),
                         ("n_ice", ice.astype(float)), ("bed_all", bed), ("bed_wet", np.where(wet, bed, 0.0)),
                         ("thick_ice", np.where(ice, thick, 0.0))]:
                sums[k] += np.bincount(idx, weights=w, minlength=NCELL)
        bounds = fbed.bounds
    sums = {k: v.reshape(J_SOUTH, NX) for k, v in sums.items()}

    # cells entirely inside the Charrassin raster (all four corners inside its bounds)
    ll_to_ps = Transformer.from_crs("EPSG:4326", "EPSG:3031", always_xy=True)
    LON, LAT = np.meshgrid(lon_edges, lat_edges)
    X, Y = ll_to_ps.transform(LON, LAT)
    inside = (X > bounds.left) & (X < bounds.right) & (Y > bounds.bottom) & (Y < bounds.top)
    sums["covered"] = inside[:-1, :-1] & inside[1:, :-1] & inside[:-1, 1:] & inside[1:, 1:] & (sums["n_all"] > 0)
    return sums

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# Charrassin_Bed_elevation_data = xr.open_dataset('/g/data/x77/cy8964/Charrassin2025_Data/BED_ANTGG2022_lat_lon.nc', decode_times=False)
#

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# #Re-using the weights created for Bedmachine_v3 to model grid, since both the Charrassin and Bedmachine_v3 has same resolution (500m*500m) and same projection EPSG3031.
# regridder_Bed_machine_to_GEBCO = xesmf.Regridder(
#     Charrassin_Bed_elevation_data,
#     topog,
#     "bilinear",
#     periodic=True,
#     filename='/g/data/x77/cy8964/mom6/input/input-8km/Regridd_Charrassin_to_8km_global_grid_Bilinear.nc',
#     reuse_weights=True)

In [ ]:
binned = bin_charrassin()
n = binned["n_all"]
with np.errstate(invalid="ignore", divide="ignore"):
    wet_fraction = binned["n_wet"] / n
    ocean_fraction = binned["n_ocean"] / n
    ice_fraction = binned["n_ice"] / n
    # depth: mean over wet pixels (open ocean or floating ice) where at least half the cell is wet. Otherwise the mean
    # bed over all pixels: the bed under grounded ice (positive where it is below sea level), which the ice-shelf
    # (cavity) topography uses to extend the ocean inland of the grounding line, and negative on land. For the
    # no-cavity topography, depth > 0 together with thick == 0 still means "at least half wet and not ice covered".
    depth_charrassin = np.where(wet_fraction >= 0.5, -binned["bed_wet"] / binned["n_wet"], -binned["bed_all"] / n)
    # ice thickness: mean over ice pixels where at least half the cell has ice, otherwise 0, so that thick > 0 still
    # means "at least half ice covered"
    thick_charrassin = np.where(ice_fraction >= 0.5, binned["thick_ice"] / binned["n_ice"], 0.0)
covered = binned["covered"]
print(f"model cells south of 40S: {covered.size}, covered by Charrassin: {covered.sum()}, "
      f"pixels per covered cell: min {n[covered].min():.0f}, median {np.median(n[covered]):.0f}")

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# Elevation = Elevation.assign_coords(lat=topog['lat'], lon=topog['lon'])
#

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# #Elevation = Elevation*0 + Elevation.values[:,::-1]

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# plt.figure(figsize=(12, 6))# You can adjust the width (12) and height (6) as needed
# Elevation.plot(x='lon', y='lat')
# plt.ylim(-87,-35)
# #plt.savefig("/g/data1a/tm70/ek4684/Bedmachine_data/elevation_from_Charrassin_on_model_grid.png", dpi=300, bbox_inches='tight')
# plt.show()

In [ ]:
# Use Charrassin wherever it covers the model cell, and the GEBCO-based global topography (topog.nc) elsewhere.
# (Previously GEBCO was used wherever the regridded Charrassin elevation was >= 0, which also replaced Charrassin land
# and Charrassin cells with an elevation of exactly 0 inside its coverage.)
def fill(charrassin_values, outside_values):
    out = np.array(outside_values, dtype=np.float64, copy=True)
    out[:J_SOUTH][covered] = charrassin_values[covered]
    return out

gebco_depth = topog.depth.fillna(0).values
gebco_fraction = topog.sea_area_fraction.fillna(0).values
depth_filled = fill(depth_charrassin, gebco_depth)
ocean_fraction_filled = fill(ocean_fraction, gebco_fraction)
wet_fraction_filled = fill(wet_fraction, gebco_fraction)
ice_fraction_filled = fill(ice_fraction, np.zeros_like(gebco_depth))
thick_filled = fill(thick_charrassin, np.zeros_like(gebco_depth))
coverage = fill(np.ones_like(depth_charrassin), np.zeros_like(gebco_depth)).astype(np.int8)

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# plt.figure(figsize=(12, 6))# You can adjust the width (12) and height (6) as needed
# (Elevation_filled).plot(x='lon', y='lat', cbar_kwargs = {'label':'bed elevation (m)'})
# plt.ylim(-87,-35)
# #plt.savefig("/g/data1a/tm70/ek4684/Bedmachine_data/elevation_from_Charrassin_on_model_grid.png", dpi=300, bbox_inches='tight')
# plt.ylabel('Latitude')
# plt.xlabel('Longitude')
# plt.title('Charrassin2025 depth filled with GEBCO2024 to the north')
# plt.show()

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# plt.figure(figsize=(12, 6))# You can adjust the width (12) and height (6) as needed
# (-Elevation_filled-topog.depth).plot(x='lon', y='lat', vmax = 100, vmin = -100, cmap = 'RdBu_r', cbar_kwargs = {'label':'depth anomaly (m)'})
# (Elevation.where(Elevation !=0)*0+1).fillna(0).plot.contour(x='lon', y='lat', levels = [0.5],colors = 'k')
# plt.ylim(-87,-35)
# #plt.savefig("/g/data1a/tm70/ek4684/Bedmachine_data/elevation_from_Charrassin_on_model_grid.png", dpi=300, bbox_inches='tight')
# plt.ylabel('Latitude')
# plt.xlabel('Longitude')
# plt.title('Charrassin2025 depth - GEBCO2024 depth')
#
# plt.show()
#

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# plt.figure(figsize=(12, 6))# You can adjust the width (12) and height (6) as needed
# (Elevation_filled).diff('ny').plot(vmax = 50, vmin = -50, cmap = 'RdBu_r', x = 'lon', y = 'lat',cbar_kwargs = {'label':'bed elevation difference between meridional cells (m)'})
# plt.ylim(-87,-35)
# #plt.savefig("/g/data1a/tm70/ek4684/Bedmachine_data/elevation_from_Charrassin_on_model_grid.png", dpi=300, bbox_inches='tight')
# plt.ylabel('Latitude')
# plt.xlabel('Longitude')
# plt.title('Charrassin2025 depth filled with GEBCO2024 to the north:\n Difference between subsequent y cells i.e. (Elevation_filled).diff("ny")')
# plt.show()

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# plt.figure(figsize=(12, 6))# You can adjust the width (12) and height (6) as needed
# (Elevation_filled).diff('nx').plot(vmax = 50, vmin = -50, cmap = 'RdBu_r', x = 'lon', y = 'lat',cbar_kwargs = {'label':'bed elevation difference between zonal cells (m)'})
# plt.ylim(-87,-35)
# #plt.savefig("/g/data1a/tm70/ek4684/Bedmachine_data/elevation_from_Charrassin_on_model_grid.png", dpi=300, bbox_inches='tight')
# plt.ylabel('Latitude')
# plt.xlabel('Longitude')
# plt.title('Charrassin2025 depth filled with GEBCO2024 to the north:\n Difference between subsequent x cells i.e. (Elevation_filled).diff("nx")')
# plt.show()

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# topog.sea_area_fraction.plot()

In [ ]:
topog_Charrassin = xr.Dataset(
    data_vars=dict(
        depth=(["ny","nx"], depth_filled.astype(np.float32), dict(units="m", long_name="depth (positive down; negative on land)")),
        ocean_fraction=(["ny","nx"], ocean_fraction_filled.astype(np.float32), dict(units="1", long_name="fraction of Charrassin pixels that are open ocean (mask 0); GEBCO sea_area_fraction outside Charrassin coverage")),
        wet_fraction=(["ny","nx"], wet_fraction_filled.astype(np.float32), dict(units="1", long_name="fraction of Charrassin pixels that are open ocean or floating ice (mask 0 or 3); GEBCO sea_area_fraction outside Charrassin coverage")),
        charrassin_coverage=(["ny","nx"], coverage, dict(long_name="1 where the cell is entirely within the Charrassin raster, 0 where filled from GEBCO")),
        lat=(["ny","nx"], topog.lat.values),
        lon=(["ny","nx"], topog.lon.values),
    ),
    attrs=dict(description='Charrassin 2025 bed (BED/MASK_ANTGG2022.tif) binned onto the 8km model grid: each cell is the mean depth of the Charrassin 500 m pixels it contains that are wet (open ocean or floating ice) if at least half of them are wet, otherwise the mean bed over all its pixels (the bed under grounded ice; negative on land). Cells not entirely within the Charrassin raster are filled with the GEBCO 2024 based global topography (/g/data/x77/ahg157/inputs/mom6/global-8km/topog.nc).', history=PROVENANCE),
)

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# topog_Charrassin 

In [ ]:
# save netcdf
topog_Charrassin.to_netcdf(f"{OUTPUT_DIR}/topog_Charrassin_regridded.nc")

# Repeat for mask and ice shelf

**[Out of date: the mask is no longer regridded. The open-ocean, wet and ice fractions now come from `bin_charrassin` (above). The code cells in this section are commented out.]**


In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# # Open dataset
# dataset = gdal.Open('/g/data/x77/cy8964/Charrassin2025_Data/MASK_ANTGG2022.tif')
# band = dataset.GetRasterBand(1)
# data = band.ReadAsArray()
#
# # Get GeoTransform and projection
# gt = dataset.GetGeoTransform()
# nx, ny = dataset.RasterXSize, dataset.RasterYSize
#
# # Build coordinates from geotransform
# x = np.arange(gt[0], gt[0] + nx * gt[1], gt[1])
# y = np.arange(gt[3], gt[3] + ny * gt[5], gt[5])
#
# # y is descending if gt[5] < 0 (common for north-up images), reverse if needed
# if gt[5] < 0:
#     y = y[::-1]
#     data = data[::-1, :]  # Flip data accordingly
#
# da = xr.DataArray(
#     data,
#     dims=('y', 'x'),
#     coords={'x': x, 'y': y},
#     attrs={
#         'crs': dataset.GetProjection(),
#         'transform': gt,
#         'nodata': band.GetNoDataValue()
#     }
# )
#
# ds_xr = xr.Dataset({'mask': da})
#
# # Create 2D coordinate grids
# X, Y = np.meshgrid(x, y)
#
# # Flatten and transform
# lon_flat, lat_flat = ps_to_ll.transform(X.ravel(), Y.ravel())
# lon_flat = (lon_flat + 280) % 360 - 280  # Normalize longitudes
#
# # Reshape to 2D
# new_longitudes = lon_flat.reshape(X.shape)
# new_latitudes = lat_flat.reshape(Y.shape)
#
# ds_out = xr.Dataset(
#     data_vars={
#         "mask": (("y", "x"), ds_xr.mask.data)
#     },
#     coords={
#         "lat": (("y", "x"), new_latitudes),
#         "lon": (("y", "x"), new_longitudes)
#     },
#     attrs={
#         "title": "Bedmachine Antarctic Topography",
#         "source_crs": "EPSG:3031",
#         "converted_to": "EPSG:4326",
#         "note": "Original Charrassin mask data was in polar stereographic projection and converted to lat/lon."
#     }
# )
#
# ds_out["mask"].attrs["units"] = "none"
# ds_out["mask"].attrs["long_name"] = "Ocean/ice/land mask, byte, 0-5, ocean (0); ice-free land (1); grounded ice (2); floating ice (3); Lake Vostok (4)"
# ds_out["lat"].attrs["units"] = "degrees_north"
# ds_out["lon"].attrs["units"] = "degrees_east"
# #ds_out.to_netcdf("/g/data/x77/cy8964/Charrassin2025_Data/MASK_ANTGG2022_lat_lon.nc")

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# Charrassin_Mask_data.mask

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# Charrassin_Mask_data = xr.open_dataset('/g/data/x77/cy8964/Charrassin2025_Data/MASK_ANTGG2022_lat_lon.nc', decode_times=False)
# #Re-using the weights created for Bedmachine_v3 to model grid, since both the Charrassin and Bedmachine_v3 has same resolution (500m*500m) and same projection EPSG3031.
# regridder_Bed_machine_to_GEBCO = xesmf.Regridder(
#     Charrassin_Mask_data,
#     topog,
#     "bilinear",
#     periodic=True,
#     filename='/g/data/x77/cy8964/mom6/input/input-8km/Regridd_Charrassin_to_8km_global_grid_Bilinear.nc',
#     reuse_weights=True)
#
# Mask = regridder_Bed_machine_to_GEBCO(Charrassin_Mask_data.mask)
# Mask = Mask.assign_coords(lat=topog['lat'], lon=topog['lon'])
# plt.figure(figsize=(12, 6))# You can adjust the width (12) and height (6) as needed
# Mask.plot(x='lon', y='lat', cbar_kwargs = {'label':"Ocean/ice/land mask, byte, 0-5, ocean (0); ice-free land (1); \n grounded ice (2); floating ice (3); Lake Vostok (4)"})
# plt.ylim(-87,-35)
# plt.ylabel('Latitude')
# plt.xlabel('Longitude')
# plt.title('Charrassin2025 mask - xesmf bilinear regridding onto model grid')
#
# #plt.savefig("/g/data1a/tm70/ek4684/Bedmachine_data/elevation_from_Charrassin_on_model_grid.png", dpi=300, bbox_inches='tight')
# plt.show()

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# plt.figure(figsize=(12, 6)) #Limit to vmax = 1: still some ocean cells in continent....
# Mask.plot(x='lon', y='lat', vmax = 1)
# plt.ylim(-87,-35)
# #plt.savefig("/g/data1a/tm70/ek4684/Bedmachine_data/elevation_from_Charrassin_on_model_grid.png", dpi=300, bbox_inches='tight')
# plt.show()

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# #load mask consistent with topog.nc
# Angus_mask = xr.open_dataset('/g/data/x77/ahg157/inputs/mom6/global-8km/ocean_mask.nc')
# Angus_mask

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# plt.figure(figsize=(12, 6))# You can adjust the width (12) and height (6) as needed
# ((xr.where(Mask>0.5,0,1))-Angus_mask.mask.values).plot(x='lon', y='lat')
# plt.ylim(-87,-35)
# #plt.savefig("/g/data1a/tm70/ek4684/Bedmachine_data/elevation_from_Charrassin_on_model_grid.png", dpi=300, bbox_inches='tight')
# plt.show()

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# np.unique(Mask.values)

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# fig = plt.figure(figsize = (10,8), dpi = 150)
# Charrassin_Mask_data.mask.plot()
# plt.title('Original Charrassin mask data')

# Try nearest neighbour xesmf method

**[Out of date: see the note at the start of this section.]**

```
mpirun -np 192 ESMF_RegridWeightGen -p none -i --ignore_degenerate -s /g/data/x77/cy8964/Charrassin2025_Data/MASK_ANTGG2022_lat_lon.nc -d /g/data/x77/ahg157/inputs/mom6/global-8km/topog.nc -w /g/data/x77/cy8964/mom6/input/input-8km/Regridd_Charrassin_to_8km_global_grid_NearestNeighbour.nc -m nearestdtos --netcdf4 --src_regional --check

```
this one resulted in values out of range - see nearest_d2s option here https://xesmf.readthedocs.io/en/latest/notebooks/Compare_algorithms.html

BUT options `neareststod` hangs for an hour, as does `conserve`. Is it really that much more expensive than bilinear?

`neareststod` command:

```
qsub -I -P x77 -q hugemem -l ncpus=192,mem=5880G,walltime=05:00:00,storage=gdata/hh5+gdata/ik11+gdata/x77+gdata/vk83
module load esmf/8.6.1 openmpi/4.1.7
mpirun -np 192 ESMF_RegridWeightGen -p none -i --ignore_degenerate -s /g/data/x77/cy8964/Charrassin2025_Data/MASK_ANTGG2022_lat_lon.nc -d /g/data/x77/ahg157/inputs/mom6/global-8km/topog.nc -w /g/data/x77/cy8964/mom6/input/input-8km/Regridd_Charrassin_to_8km_global_grid_NearestNeighbour_stod.nc -m neareststod --netcdf4 --src_regional --check

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# Charrassin_Mask_data = xr.open_dataset('/g/data/x77/cy8964/Charrassin2025_Data/MASK_ANTGG2022_lat_lon.nc', decode_times=False)
# #Re-using the weights created for Bedmachine_v3 to model grid, since both the Charrassin and Bedmachine_v3 has same resolution (500m*500m) and same projection EPSG3031.
# regridder_Bed_machine_to_GEBCO = xesmf.Regridder(
#     Charrassin_Mask_data,
#     topog,
#     "nearestdtos",
#     periodic=True,
#     filename='/g/data/x77/cy8964/mom6/input/input-8km/Regridd_Charrassin_to_8km_global_grid_NearestNeighbour.nc',
#     reuse_weights=True)
#
# Mask = regridder_Bed_machine_to_GEBCO(Charrassin_Mask_data.mask)
# Mask = Mask.assign_coords(lat=topog['lat'], lon=topog['lon'])
# plt.figure(figsize=(12, 6))# You can adjust the width (12) and height (6) as needed
# Mask.plot(x='lon', y='lat')
# plt.ylim(-87,-35)
# #plt.savefig("/g/data1a/tm70/ek4684/Bedmachine_data/elevation_from_Charrassin_on_model_grid.png", dpi=300, bbox_inches='tight')
# plt.show()

# Ice thickness: bilinear is good

**[Out of date: ice thickness is now binned in the same pass as the bed (`bin_charrassin`, above), not bilinearly regridded.]**


In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# # Open dataset
# dataset = gdal.Open('/g/data/x77/cy8964/Charrassin2025_Data/ICE_THICKNESS_ANTGG2022.tif')
# band = dataset.GetRasterBand(1)
# data = band.ReadAsArray()
#
# # Get GeoTransform and projection
# gt = dataset.GetGeoTransform()
# nx, ny = dataset.RasterXSize, dataset.RasterYSize
#
# # Build coordinates from geotransform
# x = np.arange(gt[0], gt[0] + nx * gt[1], gt[1])
# y = np.arange(gt[3], gt[3] + ny * gt[5], gt[5])
#
# # y is descending if gt[5] < 0 (common for north-up images), reverse if needed
# if gt[5] < 0:
#     y = y[::-1]
#     data = data[::-1, :]  # Flip data accordingly
#
# da = xr.DataArray(
#     data,
#     dims=('y', 'x'),
#     coords={'x': x, 'y': y},
#     attrs={
#         'crs': dataset.GetProjection(),
#         'transform': gt,
#         'nodata': band.GetNoDataValue()
#     }
# )
#
# ds_xr = xr.Dataset({'thick': da})
#
# # Create 2D coordinate grids
# X, Y = np.meshgrid(x, y)
#
# # Flatten and transform
# lon_flat, lat_flat = ps_to_ll.transform(X.ravel(), Y.ravel())
# lon_flat = (lon_flat + 280) % 360 - 280  # Normalize longitudes
#
# # Reshape to 2D
# new_longitudes = lon_flat.reshape(X.shape)
# new_latitudes = lat_flat.reshape(Y.shape)
#
# ds_out = xr.Dataset(
#     data_vars={
#         "thick": (("y", "x"), ds_xr.thick.data)
#     },
#     coords={
#         "lat": (("y", "x"), new_latitudes),
#         "lon": (("y", "x"), new_longitudes)
#     },
#     attrs={
#         "title": "Bedmachine Antarctic Topography",
#         "source_crs": "EPSG:3031",
#         "converted_to": "EPSG:4326",
#         "note": "Original Charrassin ice thickness data was in polar stereographic projection and converted to lat/lon."
#     }
# )
#
# ds_out["thick"].attrs["units"] = "meters"
# ds_out["thick"].attrs["long_name"] = "Ice thickness (meters), float, 0 to 5000 meters."
# ds_out["lat"].attrs["units"] = "degrees_north"
# ds_out["lon"].attrs["units"] = "degrees_east"
# #ds_out.to_netcdf("/g/data/x77/cy8964/Charrassin2025_Data/ICE_THICKNESS_ANTGG2022_lat_lon.nc")

In [ ]:
# Ice thickness comes from the same binning pass as the bed (bin_charrassin, above)
plt.figure(figsize=(12, 6))
plt.pcolormesh(thick_filled[:J_SOUTH])
plt.colorbar(label="ice thickness (m)")
plt.show()

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# IceThick
# "Ice thickness (meters), float, 0 to 5000 meters."

In [ ]:
ice_Charrassin = xr.Dataset(
    data_vars=dict(
        thick=(["ny","nx"], thick_filled.astype(np.float32), dict(units="m", long_name="ice thickness")),
        ice_fraction=(["ny","nx"], ice_fraction_filled.astype(np.float32), dict(units="1", long_name="fraction of Charrassin pixels with ice thickness > 0; 0 outside Charrassin coverage")),
        lat=(["ny","nx"], topog.lat.values),
        lon=(["ny","nx"], topog.lon.values),
    ),
    attrs=dict(description='Ice thickness (meters) from Charrassin 2025 (ICE_THICKNESS_ANTGG2022.tif) binned onto the 8km model grid: each cell is the mean thickness of the Charrassin 500 m pixels it contains that have ice if at least half of them do, otherwise 0. Cells not entirely within the Charrassin raster are 0.', history=PROVENANCE),
)

In [ ]:
# save netcdf
ice_Charrassin.to_netcdf(f"{OUTPUT_DIR}/ice_thickness_Charrassin_regridded.nc")

# Surface

**[Not regenerated: the surface elevation is not used for the panan (no-cavity) topography. The code cells in this section are commented out.]**


In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# # Open dataset
# dataset = gdal.Open('/g/data/x77/cy8964/Charrassin2025_Data/SURFACE_ANTGG2022.tif')
# band = dataset.GetRasterBand(1)
# data = band.ReadAsArray()
#
# # Get GeoTransform and projection
# gt = dataset.GetGeoTransform()
# nx, ny = dataset.RasterXSize, dataset.RasterYSize
#
# # Build coordinates from geotransform
# x = np.arange(gt[0], gt[0] + nx * gt[1], gt[1])
# y = np.arange(gt[3], gt[3] + ny * gt[5], gt[5])
#
# # y is descending if gt[5] < 0 (common for north-up images), reverse if needed
# if gt[5] < 0:
#     y = y[::-1]
#     data = data[::-1, :]  # Flip data accordingly
#
# da = xr.DataArray(
#     data,
#     dims=('y', 'x'),
#     coords={'x': x, 'y': y},
#     attrs={
#         'crs': dataset.GetProjection(),
#         'transform': gt,
#         'nodata': band.GetNoDataValue()
#     }
# )
#
# ds_xr = xr.Dataset({'elevation': da})
#
# # Create 2D coordinate grids
# X, Y = np.meshgrid(x, y)
#
# # Flatten and transform
# lon_flat, lat_flat = ps_to_ll.transform(X.ravel(), Y.ravel())
# lon_flat = (lon_flat + 280) % 360 - 280  # Normalize longitudes
#
# # Reshape to 2D
# new_longitudes = lon_flat.reshape(X.shape)
# new_latitudes = lat_flat.reshape(Y.shape)
#
# ds_out = xr.Dataset(
#     data_vars={
#         "elevation": (("y", "x"), ds_xr.elevation.data)
#     },
#     coords={
#         "lat": (("y", "x"), new_latitudes),
#         "lon": (("y", "x"), new_longitudes)
#     },
#     attrs={
#         "title": "Bedmachine Antarctic Topography",
#         "source_crs": "EPSG:3031",
#         "converted_to": "EPSG:4326",
#         "note": "Original Charrassin surface elevation data was in polar stereographic projection and converted to lat/lon."
#     }
# )
#
# ds_out["elevation"].attrs["units"] = "meters"
# ds_out["elevation"].attrs["long_name"] = "Surface elevation above sea level, float, 0 to 5000 meters."
# ds_out["lat"].attrs["units"] = "degrees_north"
# ds_out["lon"].attrs["units"] = "degrees_east"
# ds_out.to_netcdf("/g/data/x77/cy8964/Charrassin2025_Data/SURFACE_ANTGG2022_lat_lon.nc")

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# Charrassin_Surface_data = xr.open_dataset('/g/data/x77/cy8964/Charrassin2025_Data/SURFACE_ANTGG2022_lat_lon.nc', decode_times=False)
# #Re-using the weights created for Bedmachine_v3 to model grid, since both the Charrassin and Bedmachine_v3 has same resolution (500m*500m) and same projection EPSG3031.
# regridder_Bed_machine_to_GEBCO = xesmf.Regridder(
#     Charrassin_Surface_data,
#     topog,
#     "bilinear",
#     periodic=True,
#     filename='/g/data/x77/cy8964/mom6/input/input-8km/Regridd_Charrassin_to_8km_global_grid_Bilinear.nc',
#     reuse_weights=True)
#
# Surface = regridder_Bed_machine_to_GEBCO(Charrassin_Surface_data.elevation)
# Surface = Surface.assign_coords(lat=topog['lat'], lon=topog['lon'])
# plt.figure(figsize=(12, 6))# You can adjust the width (12) and height (6) as needed
# Surface.plot(x='lon', y='lat')
# plt.ylim(-87,-35)
# #plt.savefig("/g/data1a/tm70/ek4684/Bedmachine_data/elevation_from_Charrassin_on_model_grid.png", dpi=300, bbox_inches='tight')
# plt.show()

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# surface_Charrassin = xr.Dataset(
#     data_vars=dict(
#         elevation=(["ny","nx"], Surface.values),
#         lat=(["ny","nx"], Surface.lat.values),
#         lon=(["ny","nx"], Surface.lon.values),           
#     ),
#     attrs=dict(description="Surface elevation above sea level, float, 0 to 5000 meters. From Charrassin 2025 depth regridded onto 8km grid, filled elsewhere with GEBCO 2024 data. Generated with notebook https://github.com/claireyung/mom6-panAn-iceshelf-tools/blob/add_charrassin_bathy/generate-draft/Generate-Charrassin-bathy.ipynb "),
# )

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# # save netcdf to gdata
# surface_Charrassin.to_netcdf('/g/data/x77/cy8964/mom6/input/input-8km/surface_Charrassin_regridded.nc')

# WATER_HEIGHT

**[Not regenerated: the water height is not used for the panan (no-cavity) topography. The code cells in this section are commented out.]**


In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# # Open dataset
# dataset = gdal.Open('/g/data/x77/cy8964/Charrassin2025_Data/WATER_HEIGHT_ANTGG2022.tif')
# band = dataset.GetRasterBand(1)
# data = band.ReadAsArray()
#
# # Get GeoTransform and projection
# gt = dataset.GetGeoTransform()
# nx, ny = dataset.RasterXSize, dataset.RasterYSize
#
# # Build coordinates from geotransform
# x = np.arange(gt[0], gt[0] + nx * gt[1], gt[1])
# y = np.arange(gt[3], gt[3] + ny * gt[5], gt[5])
#
# # y is descending if gt[5] < 0 (common for north-up images), reverse if needed
# if gt[5] < 0:
#     y = y[::-1]
#     data = data[::-1, :]  # Flip data accordingly
#
# da = xr.DataArray(
#     data,
#     dims=('y', 'x'),
#     coords={'x': x, 'y': y},
#     attrs={
#         'crs': dataset.GetProjection(),
#         'transform': gt,
#         'nodata': band.GetNoDataValue()
#     }
# )
#
# ds_xr = xr.Dataset({'elevation': da})
#
# # Create 2D coordinate grids
# X, Y = np.meshgrid(x, y)
#
# # Flatten and transform
# lon_flat, lat_flat = ps_to_ll.transform(X.ravel(), Y.ravel())
# lon_flat = (lon_flat + 280) % 360 - 280  # Normalize longitudes
#
# # Reshape to 2D
# new_longitudes = lon_flat.reshape(X.shape)
# new_latitudes = lat_flat.reshape(Y.shape)
#
# ds_out = xr.Dataset(
#     data_vars={
#         "elevation": (("y", "x"), ds_xr.elevation.data)
#     },
#     coords={
#         "lat": (("y", "x"), new_latitudes),
#         "lon": (("y", "x"), new_longitudes)
#     },
#     attrs={
#         "title": "Bedmachine Antarctic Topography",
#         "source_crs": "EPSG:3031",
#         "converted_to": "EPSG:4326",
#         "note": "Original Charrassin water height elevation data was in polar stereographic projection and converted to lat/lon."
#     }
# )
#
# ds_out["elevation"].attrs["units"] = "meters"
# ds_out["elevation"].attrs["long_name"] = "Water height under the ice shelves, float, 0 to 8023 meters."
# ds_out["lat"].attrs["units"] = "degrees_north"
# ds_out["lon"].attrs["units"] = "degrees_east"
# ds_out.to_netcdf("/g/data/x77/cy8964/Charrassin2025_Data/WATER_HEIGHT_ANTGG2022_lat_lon.nc")

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# Charrassin_WH_data = xr.open_dataset('/g/data/x77/cy8964/Charrassin2025_Data/WATER_HEIGHT_ANTGG2022_lat_lon.nc', decode_times=False)
# #Re-using the weights created for Bedmachine_v3 to model grid, since both the Charrassin and Bedmachine_v3 has same resolution (500m*500m) and same projection EPSG3031.
# regridder_Bed_machine_to_GEBCO = xesmf.Regridder(
#     Charrassin_WH_data,
#     topog,
#     "bilinear",
#     periodic=True,
#     filename='/g/data/x77/cy8964/mom6/input/input-8km/Regridd_Charrassin_to_8km_global_grid_Bilinear.nc',
#     reuse_weights=True)
#
# WH = regridder_Bed_machine_to_GEBCO(Charrassin_WH_data.elevation)
# WH = WH.assign_coords(lat=topog['lat'], lon=topog['lon'])
# plt.figure(figsize=(12, 6))# You can adjust the width (12) and height (6) as needed
# WH.plot(x='lon', y='lat')
# plt.ylim(-87,-35)
# #plt.savefig("/g/data1a/tm70/ek4684/Bedmachine_data/elevation_from_Charrassin_on_model_grid.png", dpi=300, bbox_inches='tight')
# plt.show()

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# WH_Charrassin = xr.Dataset(
#     data_vars=dict(
#         elevation=(["ny","nx"], WH.values),
#         lat=(["ny","nx"], WH.lat.values),
#         lon=(["ny","nx"], WH.lon.values),           
#     ),
#     attrs=dict(description="Water height under the ice shelves, float, 0 to 8023 meters. From Charrassin 2025 depth regridded onto 8km grid, filled elsewhere with GEBCO 2024 data. Generated with notebook https://github.com/claireyung/mom6-panAn-iceshelf-tools/blob/add_charrassin_bathy/generate-draft/Generate-Charrassin-bathy.ipynb "),
# )

In [ ]:
# [commented out: not needed to make the binned topography and ice thickness]
# # save netcdf to gdata
# WH_Charrassin.to_netcdf('/g/data/x77/cy8964/mom6/input/input-8km/WATER_HEIGHT_Charrassin_regridded.nc')